In [ ]:
# @title Package installation
# --- ESSENTIALS (Keep these) ---
from scipy import sparse                     # You need this for sparse.csr_matrix
import pandas as pd                          # Essential for your dataframes
import numpy as np                           # Essential for the heavy math
from sklearn.metrics.pairwise import cosine_similarity # Essential for user similarity
import requests                              # You need this to fetch the CSVs from GitHub
import gc                                    # Essential for our new RAM-saving strategy

try:
    from google.colab import files
except ImportError:
    pass

# --- TEXT & NLP (Comment out if you aren't doing content-based filtering right now) ---
from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.decomposition import TruncatedSVD
from sentence_transformers import SentenceTransformer
# import re

In [ ]:
# @title Importing the data

# Set username and repository name
user = "miachambat"
repo = "Machine_Learning_Project_Team_Geneva"

# Use the /contents endpoint to get the file list from the root
api_url = f"https://api.github.com/repos/{user}/{repo}/contents"

response = requests.get(api_url)

# We then check if the request was successful
if response.status_code == 200:
    file_list = response.json()
    dataframes = {}

    for file in file_list:
        # Here we only keep the CSV files. We don't need the README.md for our coding.
        if file['name'].endswith('.csv'):
            file_name = file['name']
            raw_url = file['download_url']

            # Explicitly handle the Mac-exported file with a semicolon delimiter
            if file_name == 'items_enriched.csv':
                print(f"Loading {file_name} with Mac delimiter (;)...")
                dataframes[file_name] = pd.read_csv(raw_url, sep=';')
            else:
                # For all other files, try the default comma separator first
                try:
                    dataframes[file_name] = pd.read_csv(raw_url, sep=',')
                except pd.errors.ParserError:
                    # Keep a fallback just in case other files also use semicolons
                    dataframes[file_name] = pd.read_csv(raw_url, sep=';')

    print("\nSuccessfully loaded:")
    for name in dataframes.keys():
        print(f"- {name}")
else:
    print(f"Failed to fetch data. Status code: {response.status_code}")

interactions = dataframes.get('interactions_train.csv')
items = dataframes.get('items.csv')
items_enriched = dataframes.get('items_enriched.csv')


Successfully loaded:
- interactions_train.csv
- items.csv


The core difference: user-level vs. global temporal splits

The main difference between the two blocks of code boils down to whether you are ranking time per user or across the entire dataset.

1. User-level temporal split: groups the data by user ("u") and ranks the timestamps ("t") within that specific user's history. The result: A pct_rank of 0.8 means "the 80th percentile of this specific user's timeline." If you split the data here, you are training on the first 80% of every individual user's clicks, and testing on their final 20%.

2. Global temporal split: What it does: It looks at all timestamps across the entire dataframe and ranks them from oldest to newest, completely ignoring who made the click.

In [ ]:
# @title User temporal split

# Let's first sort the interactions by user and time stamp
# interactions = interactions.sort_values(["u", "t"])
# interactions["pct_rank"] = interactions.groupby("u")["t"].rank(pct=True, method='dense')
# interactions.reset_index(inplace=True, drop=True)

In [ ]:
# @title Global temporal split (more realistic)
interactions = interactions.sort_values("t")
interactions["pct_rank"] = interactions["t"].rank(pct=True, method='dense')
train_data = interactions.copy()
train_users = set(train_data['u'].unique())

In [ ]:
# @title Data matrix

n_users = interactions.u.nunique()
n_items = items.i.nunique()

def create_data_matrix(data, n_users, n_items):
    """
    This function returns a numpy matrix with shape (n_users, n_items).
    Each entry is a binary value indicating positive interaction.
    """
    data_matrix = np.zeros((n_users, n_items))
    data_matrix[data["u"].values, data["i"].values] = 1
    return data_matrix

# Create the training matrix
train_data_matrix = create_data_matrix(train_data, n_users, n_items)

# Convert training matrix to Compressed Sparse Row matrices to use less data
train_data_matrix = sparse.csr_matrix(train_data_matrix)

In [ ]:
# @title Add time decay to data matrix

# 1. Create a fresh copy to avoid setting-with-copy warnings
train_data_decay = train_data.copy()

# 2. Calculate a PERSONALIZED time rank for each user
# This maps each user's timeline on a scale from 0.0 (oldest) to 1.0 (newest)
train_data_decay['user_time_rank'] = train_data_decay.groupby('u')['t'].rank(pct=True, method='dense')

# 3. Apply the decay formula
# We decay the weights down to a minimum of 0.2.
# (You can play with this minimum later: 0.5 is a gentle decay, 0.1 is aggressive!)
min_weight = 0.2
train_data_decay['decay_weight'] = min_weight + ((1.0 - min_weight) * train_data_decay['user_time_rank'])

# 4. Redefine our matrix builder to use these new weights instead of binary 1s
def create_weighted_data_matrix(data, n_users, n_items):
    data_matrix = np.zeros((n_users, n_items))
    # Instead of placing a '1', we place the exact calculated decay weight!
    data_matrix[data["u"].values, data["i"].values] = data["decay_weight"].values
    return data_matrix

print("Building the time-weighted interaction matrix...")
train_data_matrix_decay = create_weighted_data_matrix(train_data_decay, n_users, n_items)
train_data_matrix_decay = sparse.csr_matrix(train_data_matrix_decay)

print("Time-weighted matrix built successfully!")

Building the time-weighted interaction matrix...
Time-weighted matrix built successfully!


In [ ]:
# @title Basic item sim

# Define the function to predict interactions based on item similarity
def item_based_predict(train_matrix, item_sim_matrix, filename, batch_size=250):
    """
    Predicts user-item interactions based on item-item similarity.
    Parameters:
        interactions (numpy array): The user-item interaction matrix.
        similarity (numpy array): The item-item similarity matrix.
        epsilon (float): Small constant added to the denominator to avoid division by zero.
    Returns:
        numpy array: The predicted interaction scores for each user-item pair.
    """
    n_users = train_matrix.shape[0]
    all_recommendations = []

    print(f"Starting batched item-based predictions for {n_users} users...")

    # Pre-calculate the denominator (sum of similarities for each item)
    # We do this once upfront to save processing power
    item_sim_sum = np.abs(item_sim_matrix).sum(axis=1) + 1e-9

    for start_idx in range(0, n_users, batch_size):
        end_idx = min(start_idx + batch_size, n_users)

        # 1. Grab just a small batch of users
        train_batch = train_matrix[start_idx:end_idx]

        # 2. Predict for this batch
        # sparse (Users, Items) dot dense (Items, Items) = dense (Users, Items)
        # This gives us the exact same math as your old transpose method, but safely!
        pred_batch = train_batch.dot(item_sim_matrix)

        # 3. Divide by the similarity sum to normalize
        pred_batch = pred_batch / item_sim_sum

        # 4. Find the top 10 items for these users
        top_10_indices = np.argsort(pred_batch, axis=1)[:, -10:][:, ::-1]
        recs = [" ".join(map(str, row)) for row in top_10_indices]
        all_recommendations.extend(recs)

        # 5. Aggressive RAM cleanup for the loop
        del train_batch
        del pred_batch
        del top_10_indices
        gc.collect()


    # 6. Create the final submission file
    print("\nBuilding submission file...")
    df_submission = pd.DataFrame({
        'user_id': range(len(all_recommendations)),
        'recommendation': all_recommendations
    })

    df_submission.to_csv(filename, index=False)
    print(f"File saved successfully: {filename}")

    return df_submission

# Compute the item-item similarity matrix

item_similarity = cosine_similarity(train_data_matrix.T)

# Prediction and submission

# i2i_submission = item_based_predict(
#     train_matrix=train_data_matrix,
#     item_sim_matrix=item_similarity,
#     filename='i2i_submission.csv',
#     batch_size=500
# )

# files.download('i2i_submission.csv')

# del item_similarity
# gc.collect()

In [ ]:
# @title Basic user sim

# Define the function to predict interactions based on user similarity

def safe_user_based_predict_and_submit(train_matrix, user_sim_matrix, filename, batch_size=250):
    n_users = train_matrix.shape[0]
    all_recommendations = []

    print(f"Starting batched user-based predictions for {n_users} users...")

    for start_idx in range(0, n_users, batch_size):
        end_idx = min(start_idx + batch_size, n_users)

        # 1. Grab just the similarity scores for this tiny batch of users
        sim_batch = user_sim_matrix[start_idx:end_idx]

        # Compress the batch into a sparse matrix before doing the math
        sim_batch_sparse = sparse.csr_matrix(sim_batch)

        # 2. Predict just for this batch (sparse dot sparse = highly memory efficient)
        weighted_sum = sim_batch_sparse.dot(train_matrix)

        # 3. Divide by the similarity sum to normalize
        sum_sim = np.array(sim_batch_sparse.sum(axis=1)) + 1e-9

        if hasattr(weighted_sum, 'toarray'):
            pred_batch = weighted_sum.toarray() / sum_sim
        else:
            pred_batch = weighted_sum / sum_sim

        # 4. Find the top 10 items for these specific users right now
        top_10_indices = np.argsort(pred_batch, axis=1)[:, -10:][:, ::-1]
        recs = [" ".join(map(str, row)) for row in top_10_indices]
        all_recommendations.extend(recs)

        # 5. Total RAM cleanup for this batch (throw the heavy math in the trash)
        del sim_batch
        del sim_batch_sparse
        del weighted_sum
        del pred_batch
        del top_10_indices
        gc.collect()

    # 6. Create the final submission file
    print("\nBuilding submission file...")
    df_submission = pd.DataFrame({
        'user_id': range(len(all_recommendations)),
        'recommendation': all_recommendations
    })

    df_submission.to_csv(filename, index=False)
    print(f"File saved successfully: {filename}")

    return df_submission


# Compute the user-user similarity matrix
user_similarity = cosine_similarity(train_data_matrix)

# u2u_submission = safe_user_based_predict_and_submit(
#     train_matrix=train_data_matrix,
#     user_sim_matrix=user_similarity,
#     filename='u2u_submission_safe.csv',
#     batch_size=250
# )

# files.download('u2u_submission_safe.csv')

# # Cleaning up the big matrices
# del user_similarity
# gc.collect()

In [ ]:
# @title User with knn

def ultimate_safe_predict_and_submit(train_matrix, filename, k=50, batch_size=250):
    n_users = train_matrix.shape[0]
    all_recommendations = []

    print(f"Starting ultimate batched predictions for {n_users} users...")

    for start_idx in range(0, n_users, batch_size):
        end_idx = min(start_idx + batch_size, n_users)
        print(f"Processing users {start_idx} to {end_idx}...")

        # 1. Grab just the interactions for this small batch of users
        train_batch = train_matrix[start_idx:end_idx]

        # 2. Compute similarity ONLY for this batch against everyone else
        sim_batch = cosine_similarity(train_batch, train_matrix)

        # 3. Apply the KNN filter inplace directly to this small batch
        for i in range(sim_batch.shape[0]):
            row = sim_batch[i]
            if len(row) > k + 1:
                threshold_idx = np.argpartition(row, -(k+1))[-(k+1)]
                threshold_val = row[threshold_idx]
                row[row < threshold_val] = 0

        # Compress the newly zeroed-out batch into a sparse matrix!
        # This stops Numpy from un-compressing the main train_matrix in the next step.
        sim_batch_sparse = sparse.csr_matrix(sim_batch)

        # 4. Predict just for this batch (sparse dot sparse = highly memory efficient)
        weighted_sum = sim_batch_sparse.dot(train_matrix)

        # For sparse matrices, .sum(axis=1) returns a 2D matrix, so we convert it to an array
        sum_sim = np.array(sim_batch_sparse.sum(axis=1)) + 1e-9

        if hasattr(weighted_sum, 'toarray'):
            pred_batch = weighted_sum.toarray() / sum_sim
        else:
            pred_batch = weighted_sum / sum_sim

        # 5. Mask the items these specific users have already interacted with
        if hasattr(train_batch, 'toarray'):
            train_batch_dense = train_batch.toarray()
        else:
            train_batch_dense = train_batch

        pred_batch[train_batch_dense > 0] = -999

        # 6. Find the top 10 items for these users
        top_10_indices = np.argsort(pred_batch, axis=1)[:, -10:][:, ::-1]
        recs = [" ".join(map(str, row)) for row in top_10_indices]
        all_recommendations.extend(recs)

        # 7. Total RAM cleanup for this batch
        del sim_batch
        del sim_batch_sparse
        del weighted_sum
        del pred_batch
        del train_batch_dense
        del top_10_indices
        gc.collect()

    # 8. Create the final submission file
    print("\nBuilding submission file...")
    df_submission = pd.DataFrame({
        'user_id': range(len(all_recommendations)),
        'recommendation': all_recommendations
    })

    df_submission.to_csv(filename, index=False)
    print(f"File saved successfully: {filename}")

    return df_submission

# --- EXECUTE THE CODE ---
# u2u_submission_knn = ultimate_safe_predict_and_submit(
#     train_matrix=train_data_matrix,
#     filename='u2u_submission_knn.csv',
#     k=50,
#     batch_size=250
# )

# #files.download('u2u_submission_knn.csv')

# del u2u_submission_knn
# gc.collect()

In [ ]:
# @title User with popularity fallback

def u2u_with_popularity_fallback(train_matrix, filename, k=50, batch_size=250):
    n_users = train_matrix.shape[0]
    all_recommendations = []

    print("Calculating global item popularity for cold-start users...")
    # Find the top 10 most popular items across the entire training set
    item_popularity = np.array(train_matrix.sum(axis=0)).flatten()
    top_global_items = np.argsort(item_popularity)[-10:][::-1]
    top_global_string = " ".join(map(str, top_global_items))

    print(f"Starting batched predictions for {n_users} users...")

    for start_idx in range(0, n_users, batch_size):
        end_idx = min(start_idx + batch_size, n_users)

        # 1. Grab interactions for this batch
        train_batch = train_matrix[start_idx:end_idx]

        # 2. Compute similarity for this batch
        sim_batch = cosine_similarity(train_batch, train_matrix)

        # 3. Apply the KNN filter
        for i in range(sim_batch.shape[0]):
            row = sim_batch[i]
            if len(row) > k + 1:
                threshold_idx = np.argpartition(row, -(k+1))[-(k+1)]
                threshold_val = row[threshold_idx]
                row[row < threshold_val] = 0

        # 4. Compress to sparse and predict
        sim_batch_sparse = sparse.csr_matrix(sim_batch)
        weighted_sum = sim_batch_sparse.dot(train_matrix)
        sum_sim = np.array(sim_batch_sparse.sum(axis=1)) + 1e-9

        if hasattr(weighted_sum, 'toarray'):
            pred_batch = weighted_sum.toarray() / sum_sim
        else:
            pred_batch = weighted_sum / sum_sim

        # 5. Find the top 10 items for these users
        top_10_indices = np.argsort(pred_batch, axis=1)[:, -10:][:, ::-1]

        # 6. Apply the popularity fallback!
        for i in range(top_10_indices.shape[0]):
            # If the highest predicted score for this user is exactly 0,
            # it means they have no history and no neighbors.
            if np.max(pred_batch[i]) == 0:
                all_recommendations.append(top_global_string)
            else:
                recs = " ".join(map(str, top_10_indices[i]))
                all_recommendations.append(recs)

        # 7. RAM cleanup
        del sim_batch
        del sim_batch_sparse
        del weighted_sum
        del pred_batch
        del top_10_indices
        gc.collect()

    print("\nBuilding submission file...")
    df_submission = pd.DataFrame({
        'user_id': range(len(all_recommendations)),
        'recommendation': all_recommendations
    })

    df_submission.to_csv(filename, index=False)
    print(f"File saved successfully: {filename}")

    return df_submission

# --- EXECUTE THE CODE ---
# Let's run the U2U model that gave you 0.14, but with the new safety net!
# u2u_submission_pop = u2u_with_popularity_fallback(
#     train_matrix=train_data_matrix,
#     filename='u2u_submission_pop.csv',
#     k=50,
#     batch_size=250
# )

# files.download('u2u_submission_pop.csv')

In [ ]:
# @title Item with popularity fallback

def ensemble_predict_and_submit(train_matrix, item_sim_matrix, filename, k=50, batch_size=250, u2u_weight=0.5):
    n_users = train_matrix.shape[0]
    all_recommendations = []

    print("Calculating global item popularity for cold-start users...")
    item_popularity = np.array(train_matrix.sum(axis=0)).flatten()
    top_global_items = np.argsort(item_popularity)[-10:][::-1]
    top_global_string = " ".join(map(str, top_global_items))

    # Pre-calculate the Item-to-Item denominator once to save time
    item_sim_sum = np.abs(item_sim_matrix).sum(axis=1) + 1e-9

    print(f"Starting batched Ensemble predictions for {n_users} users...")

    for start_idx in range(0, n_users, batch_size):
        end_idx = min(start_idx + batch_size, n_users)

        # Grab interactions for this batch
        train_batch = train_matrix[start_idx:end_idx]

        # ==========================================
        # 1. USER-TO-USER PREDICTIONS (With KNN)
        # ==========================================
        u2u_sim_batch = cosine_similarity(train_batch, train_matrix)
        for i in range(u2u_sim_batch.shape[0]):
            row = u2u_sim_batch[i]
            if len(row) > k + 1:
                threshold_idx = np.argpartition(row, -(k+1))[-(k+1)]
                threshold_val = row[threshold_idx]
                row[row < threshold_val] = 0

        u2u_sim_sparse = sparse.csr_matrix(u2u_sim_batch)
        u2u_weighted_sum = u2u_sim_sparse.dot(train_matrix)
        u2u_sum_sim = np.array(u2u_sim_sparse.sum(axis=1)) + 1e-9

        if hasattr(u2u_weighted_sum, 'toarray'):
            u2u_pred = u2u_weighted_sum.toarray() / u2u_sum_sim
        else:
            u2u_pred = u2u_weighted_sum / u2u_sum_sim

        # ==========================================
        # 2. ITEM-TO-ITEM PREDICTIONS
        # ==========================================
        i2i_weighted_sum = train_batch.dot(item_sim_matrix)
        i2i_pred = i2i_weighted_sum / item_sim_sum

        # ==========================================
        # 3. BLEND THE MODELS TOGETHER
        # ==========================================
        i2i_weight = 1.0 - u2u_weight
        ensemble_pred = (u2u_pred * u2u_weight) + (i2i_pred * i2i_weight)

        # ==========================================
        # 4. TOP 10 & POPULARITY FALLBACK
        # ==========================================
        top_10_indices = np.argsort(ensemble_pred, axis=1)[:, -10:][:, ::-1]

        for i in range(top_10_indices.shape[0]):
            # If the highest ensemble score is 0, they have no history! Give them the popular hits.
            if np.max(ensemble_pred[i]) == 0:
                all_recommendations.append(top_global_string)
            else:
                recs = " ".join(map(str, top_10_indices[i]))
                all_recommendations.append(recs)

        # ==========================================
        # 5. AGGRESSIVE RAM CLEANUP
        # ==========================================
        del u2u_sim_batch, u2u_sim_sparse, u2u_weighted_sum, u2u_pred
        del i2i_weighted_sum, i2i_pred, ensemble_pred, top_10_indices
        gc.collect()

    print("\nBuilding submission file...")
    df_submission = pd.DataFrame({
        'user_id': range(len(all_recommendations)),
        'recommendation': all_recommendations
    })

    df_submission.to_csv(filename, index=False)
    print(f"File saved successfully: {filename}")

    return df_submission

# --- EXECUTE THE CODE ---
# Make sure you calculate the global item_similarity matrix first!
# print("Computing global item similarity...")
# item_similarity = cosine_similarity(train_data_matrix.T)

# ensemble_submission = ensemble_predict_and_submit(
#     train_matrix=train_data_matrix,
#     item_sim_matrix=item_similarity,
#     filename='ensemble_submission.csv',
#     k=50,
#     batch_size=250,
#     u2u_weight=0.5 # 0.5 means a perfect 50/50 split between both models
# )

# # files.download('ensemble_submission.csv')

# # Clean up
# gc.collect()

In [ ]:
# @title TFID (note: not as good as embedding)

# print("Building content similarity matrix...")

# # 1. Sort the items so row 0 is exactly item 0, ensuring it aligns perfectly with our train_matrix!
# items_enriched_sorted = items_enriched.sort_values('i').reset_index(drop=True)

# # 2. Fill any missing values with an empty string so the math doesn't crash
# titles = items_enriched_sorted['Title'].fillna("")
# authors = items_enriched_sorted['Author'].fillna("")
# subjects = items_enriched_sorted['Subjects'].fillna("")

# # 3. Combine Title, Author, and Subjects into one "mega-string" for the NLP engine
# text_data = titles + " " + authors + " " + subjects

# # 4. Convert the text to math
# # We limit it to 5000 features (words) to keep your Colab RAM perfectly safe
# # (Note: we leave stop_words='english' out since your dataset seems to be in French!)
# tfidf = TfidfVectorizer(max_features=5000)
# tfidf_matrix = tfidf.fit_transform(text_data)

# # 5. Calculate similarity based strictly on the content!
# content_similarity = cosine_similarity(tfidf_matrix)

# print("Content similarity matrix successfully built!")

In [ ]:
# @title First hybrid: 0.1644

def ultimate_3way_ensemble(train_matrix, item_sim_matrix, content_sim_matrix, filename,
                           k=50, batch_size=250,
                           u2u_weight=0.4, i2i_weight=0.4, content_weight=0.2):

    n_users = train_matrix.shape[0]
    all_recommendations = []

    print("Calculating global item popularity for cold-start users...")
    item_popularity = np.array(train_matrix.sum(axis=0)).flatten()
    top_global_items = np.argsort(item_popularity)[-10:][::-1]
    top_global_string = " ".join(map(str, top_global_items))

    # Pre-calculate denominators to save loop time
    item_sim_sum = np.abs(item_sim_matrix).sum(axis=1) + 1e-9
    content_sim_sum = np.abs(content_sim_matrix).sum(axis=1) + 1e-9

    print(f"Starting 3-way ensemble predictions for {n_users} users...")

    for start_idx in range(0, n_users, batch_size):
        end_idx = min(start_idx + batch_size, n_users)
        train_batch = train_matrix[start_idx:end_idx]

        # ==========================================
        # 1. USER-TO-USER
        # ==========================================
        u2u_sim_batch = cosine_similarity(train_batch, train_matrix)
        for i in range(u2u_sim_batch.shape[0]):
            row = u2u_sim_batch[i]
            if len(row) > k + 1:
                threshold_idx = np.argpartition(row, -(k+1))[-(k+1)]
                threshold_val = row[threshold_idx]
                row[row < threshold_val] = 0

        u2u_sim_sparse = sparse.csr_matrix(u2u_sim_batch)
        u2u_weighted_sum = u2u_sim_sparse.dot(train_matrix)
        u2u_sum_sim = np.array(u2u_sim_sparse.sum(axis=1)) + 1e-9

        if hasattr(u2u_weighted_sum, 'toarray'):
            u2u_pred = u2u_weighted_sum.toarray() / u2u_sum_sim
        else:
            u2u_pred = u2u_weighted_sum / u2u_sum_sim

        # ==========================================
        # 2. ITEM-TO-ITEM
        # ==========================================
        i2i_weighted_sum = train_batch.dot(item_sim_matrix)
        i2i_pred = i2i_weighted_sum / item_sim_sum

        # ==========================================
        # 3. CONTENT-BASED
        # ==========================================
        content_weighted_sum = train_batch.dot(content_sim_matrix)
        content_pred = content_weighted_sum / content_sim_sum

        # ==========================================
        # 4. THE ULTIMATE BLEND
        # ==========================================
        # We blend all three brains together based on the weights!
        ensemble_pred = (u2u_pred * u2u_weight) + (i2i_pred * i2i_weight) + (content_pred * content_weight)

        # ==========================================
        # 5. TOP 10 & POPULARITY FALLBACK
        # ==========================================
        top_10_indices = np.argsort(ensemble_pred, axis=1)[:, -10:][:, ::-1]

        for i in range(top_10_indices.shape[0]):
            if np.max(ensemble_pred[i]) == 0:
                all_recommendations.append(top_global_string)
            else:
                recs = " ".join(map(str, top_10_indices[i]))
                all_recommendations.append(recs)

        # ==========================================
        # 6. RAM CLEANUP
        # ==========================================
        del u2u_sim_batch, u2u_sim_sparse, u2u_weighted_sum, u2u_pred
        del i2i_weighted_sum, i2i_pred
        del content_weighted_sum, content_pred
        del ensemble_pred, top_10_indices
        gc.collect()

    print("\nBuilding submission file...")
    df_submission = pd.DataFrame({
        'user_id': range(len(all_recommendations)),
        'recommendation': all_recommendations
    })

    df_submission.to_csv(filename, index=False)
    print(f"File saved successfully: {filename}")

    return df_submission


# --- EXECUTE THE CODE ---
# ensemble_submission = ultimate_3way_ensemble(
#     train_matrix=train_data_matrix,
#     item_sim_matrix=item_similarity,
#     content_sim_matrix=content_similarity, # Pass our new text matrix here!
#     filename='3way_ensemble_submission.csv',
#     k=50,
#     batch_size=250,
#     u2u_weight=0.5,
#     i2i_weight=0.25,
#     content_weight=0.25
# )

try:
    from google.colab import files
    files.download('3way_ensemble_submission.csv')
except:
    pass

# Weights 0.4 0.4 0.2 --> 0.1644

Changes to implement to finetune this first hybrid:
* The sparse matrix safety upgrade
* The item noise filter (Item KNN)
* Personalized time decay

In [ ]:
# @title Adding knn for user to user

def ultimate_3way_ensemble(train_matrix, item_sim_matrix, content_sim_matrix, filename,
                           k=50, batch_size=250,
                           u2u_weight=0.4, i2i_weight=0.4, content_weight=0.2):

    n_users = train_matrix.shape[0]
    all_recommendations = []

    print("Calculating global item popularity for cold-start users...")
    item_popularity = np.array(train_matrix.sum(axis=0)).flatten()
    top_global_items = np.argsort(item_popularity)[-10:][::-1]
    top_global_string = " ".join(map(str, top_global_items))

    # --- UPDATED: Pre-calculate denominators safely for sparse matrices ---
    if hasattr(item_sim_matrix, 'toarray'):
        item_sim_sum = np.array(np.abs(item_sim_matrix).sum(axis=1)).flatten() + 1e-9
    else:
        item_sim_sum = np.array(np.abs(item_sim_matrix).sum(axis=1)).flatten() + 1e-9

    content_sim_sum = np.array(np.abs(content_sim_matrix).sum(axis=1)).flatten() + 1e-9

    print(f"Starting 3-way ensemble predictions for {n_users} users...")

    for start_idx in range(0, n_users, batch_size):
        end_idx = min(start_idx + batch_size, n_users)
        train_batch = train_matrix[start_idx:end_idx]

        # ==========================================
        # 1. USER-TO-USER
        # ==========================================
        u2u_sim_batch = cosine_similarity(train_batch, train_matrix)
        for i in range(u2u_sim_batch.shape[0]):
            row = u2u_sim_batch[i]
            if len(row) > k + 1:
                threshold_idx = np.argpartition(row, -(k+1))[-(k+1)]
                threshold_val = row[threshold_idx]
                row[row < threshold_val] = 0

        u2u_sim_sparse = sparse.csr_matrix(u2u_sim_batch)
        u2u_weighted_sum = u2u_sim_sparse.dot(train_matrix)
        u2u_sum_sim = np.array(u2u_sim_sparse.sum(axis=1)).flatten() + 1e-9

        if hasattr(u2u_weighted_sum, 'toarray'):
            u2u_pred = u2u_weighted_sum.toarray() / u2u_sum_sim[:, None]
        else:
            u2u_pred = u2u_weighted_sum / u2u_sum_sim[:, None]

        # ==========================================
        # 2. ITEM-TO-ITEM (Updated for sparse handling)
        # ==========================================
        i2i_weighted_sum = train_batch.dot(item_sim_matrix)

        if hasattr(i2i_weighted_sum, 'toarray'):
            i2i_pred = i2i_weighted_sum.toarray() / item_sim_sum
        else:
            i2i_pred = i2i_weighted_sum / item_sim_sum

        # ==========================================
        # 3. CONTENT-BASED
        # ==========================================
        content_weighted_sum = train_batch.dot(content_sim_matrix)

        if hasattr(content_weighted_sum, 'toarray'):
            content_pred = content_weighted_sum.toarray() / content_sim_sum
        else:
            content_pred = content_weighted_sum / content_sim_sum

        # ==========================================
        # 4. THE ULTIMATE BLEND
        # ==========================================
        ensemble_pred = (u2u_pred * u2u_weight) + (i2i_pred * i2i_weight) + (content_pred * content_weight)

        # ==========================================
        # 5. TOP 10 & POPULARITY FALLBACK
        # ==========================================
        top_10_indices = np.argsort(ensemble_pred, axis=1)[:, -10:][:, ::-1]

        for i in range(top_10_indices.shape[0]):
            if np.max(ensemble_pred[i]) == 0:
                all_recommendations.append(top_global_string)
            else:
                recs = " ".join(map(str, top_10_indices[i]))
                all_recommendations.append(recs)

        # ==========================================
        # 6. RAM CLEANUP
        # ==========================================
        del u2u_sim_batch, u2u_sim_sparse, u2u_weighted_sum, u2u_pred
        del i2i_weighted_sum, i2i_pred
        del content_weighted_sum, content_pred
        del ensemble_pred, top_10_indices
        gc.collect()

    print("\nBuilding submission file...")
    df_submission = pd.DataFrame({
        'user_id': range(len(all_recommendations)),
        'recommendation': all_recommendations
    })

    df_submission.to_csv(filename, index=False)
    print(f"File saved successfully: {filename}")

    return df_submission


    # Apply knn and sparse

    # --- HELPER FUNCTION ---
def apply_knn_inplace(sim_matrix, k=50):
    print(f"Applying KNN filter (k={k}) to eliminate noise...")
    for i in range(sim_matrix.shape[0]):
        row = sim_matrix[i]
        if len(row) > k + 1:
            # Find the threshold value for the top k elements
            threshold_idx = np.argpartition(row, -(k+1))[-(k+1)]
            threshold_val = row[threshold_idx]
            # Zero out anything below the threshold
            row[row < threshold_val] = 0
    return sim_matrix


# --- EXECUTE THE CODE ---
print("Computing global item similarity...")
item_similarity = cosine_similarity(train_data_matrix.T)

# 1. Filter out the noise! Only keep the top 50 closest items
# (You can tweak k=50 to k=20 later if you want it even stricter)
item_similarity = apply_knn_inplace(item_similarity, k=50)

# 2. CRITICAL MEMORY FIX: Compress the newly zeroed-out matrix!
item_similarity_sparse = sparse.csr_matrix(item_similarity)

# Free up the giant dense matrix immediately
del item_similarity
gc.collect()

# 3. Run the 3-way ensemble with the clean, compressed item matrix
# ensemble_submission = ultimate_3way_ensemble(
#     train_matrix=train_data_matrix,
#     item_sim_matrix=item_similarity_sparse, # Pass the sparse version!
#     content_sim_matrix=content_similarity,
#     filename='3way_ensemble_item_knn.csv',
#     k=50,
#     batch_size=250,
#     u2u_weight=0.4,
#     i2i_weight=0.4,
#     content_weight=0.2
# )

try:
    from google.colab import files
    files.download('3way_ensemble_item_knn.csv')
except:
    pass

Computing global item similarity...
Applying KNN filter (k=50) to eliminate noise...


In [ ]:
# @title Embedding and estimating content similarity

print("Downloading the heavy-duty multilingual AI brain...")
# Swapped to the smarter 'mpnet' model for higher accuracy
model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

print("Preparing the structured book data...")
items_sorted = items.sort_values('i').reset_index(drop=True)

titles = items_sorted['Title'].fillna("")
authors = items_sorted['Author'].fillna("")
subjects = items_sorted['Subjects'].fillna("")

# 1. THE CONTEXT UPGRADE: We format it like a mini-sentence so the AI understands it perfectly
text_data = "Titre: " + titles + ". Auteur: " + authors + ". Sujets: " + subjects + "."

print(f"Translating {len(text_data)} books into pure mathematical meaning...")
embeddings = model.encode(text_data.tolist(), show_progress_bar=True)

# Save the raw embeddings so you never have to run the model again
print("Saving embeddings to disk...")
np.save('my_smart_embeddings.npy', embeddings)

try:
    from google.colab import files
    print("Downloading file...")
    files.download('my_smart_embeddings.npy')
except:
    pass

print("Calculating semantic similarities...")
content_similarity_dense = cosine_similarity(embeddings)

# 2. THE RAM UPGRADE: We use your helper function to filter out the static noise!
# (Assuming apply_knn_inplace is already defined in your notebook)
print("Filtering out text static noise with KNN...")
content_similarity_filtered = apply_knn_inplace(content_similarity_dense, k=50)

print("Compressing the AI text matrix...")
content_similarity_sparse = sparse.csr_matrix(content_similarity_filtered)

print("AI content similarity sparse matrix successfully built!")

# RAM Cleanup
del embeddings
del content_similarity_dense
del content_similarity_filtered
gc.collect()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Preparing the structured book data...
Translating 15291 books into pure mathematical meaning...


Batches:   0%|          | 0/478 [00:00<?, ?it/s]

In [ ]:

# @title Item similarity with decay

# We use train_data_matrix_decay which inherently has the user-time decay!
item_similarity_dense = cosine_similarity(train_data_matrix_decay.T)

# Filter and compress
item_similarity_dense = apply_knn_inplace(item_similarity_dense, k=50)

print("Compressing into sparse matrix to save RAM...")
item_sim_sparse_decay = sparse.csr_matrix(item_similarity_dense)

# Immediately delete the heavy dense matrix
del item_similarity_dense
gc.collect()

In [ ]:
# @title Embeddings model: top accuracy
def hyper_fast_weight_search(train_matrix, item_sim_matrix, content_sim_matrix, weight_combos, k=50, batch_size=250):
    n_users = train_matrix.shape[0]

    # Dictionary to hold the lists of recommendations for EACH weight combo
    all_recs = {weights: [] for weights in weight_combos}

    print("Calculating global item popularity for cold-start users...")
    item_popularity = np.array(train_matrix.sum(axis=0)).flatten()
    top_global_items = np.argsort(item_popularity)[-10:][::-1]
    top_global_string = " ".join(map(str, top_global_items))

    # Pre-calculate denominators safely for sparse matrices
    if hasattr(item_sim_matrix, 'toarray'):
        item_sim_sum = np.array(np.abs(item_sim_matrix).sum(axis=1)).flatten() + 1e-9
    else:
        item_sim_sum = np.array(np.abs(item_sim_matrix).sum(axis=1)).flatten() + 1e-9

    content_sim_sum = np.array(np.abs(content_sim_matrix).sum(axis=1)).flatten() + 1e-9

    print(f"Starting hyper-fast grid search for {len(weight_combos)} combinations...")

    for start_idx in range(0, n_users, batch_size):
        end_idx = min(start_idx + batch_size, n_users)
        train_batch = train_matrix[start_idx:end_idx]

        # ==========================================
        # 1. THE HEAVY MATH (Calculated ONLY ONCE per batch)
        # ==========================================
        # U2U
        u2u_sim_batch = cosine_similarity(train_batch, train_matrix)
        for i in range(u2u_sim_batch.shape[0]):
            row = u2u_sim_batch[i]
            if len(row) > k + 1:
                threshold_idx = np.argpartition(row, -(k+1))[-(k+1)]
                threshold_val = row[threshold_idx]
                row[row < threshold_val] = 0

        u2u_sim_sparse = sparse.csr_matrix(u2u_sim_batch)
        u2u_weighted_sum = u2u_sim_sparse.dot(train_matrix)
        u2u_sum_sim = np.array(u2u_sim_sparse.sum(axis=1)).flatten() + 1e-9

        if hasattr(u2u_weighted_sum, 'toarray'):
            u2u_pred = u2u_weighted_sum.toarray() / u2u_sum_sim[:, None]
        else:
            u2u_pred = u2u_weighted_sum / u2u_sum_sim[:, None]

        # I2I
        i2i_weighted_sum = train_batch.dot(item_sim_matrix)
        if hasattr(i2i_weighted_sum, 'toarray'):
            i2i_pred = i2i_weighted_sum.toarray() / item_sim_sum
        else:
            i2i_pred = i2i_weighted_sum / item_sim_sum



        # CONTENT (AI Embeddings)
        content_weighted_sum = train_batch.dot(content_sim_matrix)
        if hasattr(content_weighted_sum, 'toarray'):
            content_pred = content_weighted_sum.toarray() / content_sim_sum
        else:
            content_pred = content_weighted_sum / content_sim_sum

        # ==========================================
        # 2. THE FAST WEIGHT LOOP
        # ==========================================
        for (w_u, w_i, w_c) in weight_combos:

            ensemble_pred = (u2u_pred * w_u) + (i2i_pred * w_i) + (content_pred * w_c)
            top_10_indices = np.argsort(ensemble_pred, axis=1)[:, -10:][:, ::-1]

            for i in range(top_10_indices.shape[0]):
                if np.max(ensemble_pred[i]) == 0:
                    all_recs[(w_u, w_i, w_c)].append(top_global_string)
                else:
                    recs = " ".join(map(str, top_10_indices[i]))
                    all_recs[(w_u, w_i, w_c)].append(recs)

        # ==========================================
        # 3. RAM CLEANUP
        # ==========================================
        del u2u_sim_batch, u2u_sim_sparse, u2u_weighted_sum, u2u_pred
        del i2i_weighted_sum, i2i_pred, content_weighted_sum, content_pred
        try: del ensemble_pred, top_10_indices
        except: pass
        gc.collect()

    print("\nBuilding submission files...")
    for weights, recs in all_recs.items():
        filename = f"ai_ensemble_{weights[0]}_{weights[1]}_{weights[2]}.csv"
        df_submission = pd.DataFrame({
            'user_id': range(len(recs)),
            'recommendation': recs
        })
        df_submission.to_csv(filename, index=False)
        print(f"File saved successfully: {filename}")

        # Try to trigger the download!
        try:
            files.download(filename)
        except:
            pass

# We assume train_data_matrix_decay, item_sim_sparse_decay, and content_similarity (the embeddings) exist!

my_weight_tests = [
    (0.4, 0.4, 0.2),  # The old baseline (just to see how much the AI improves it alone)
    (0.35, 0.35, 0.3) ] # Giving the AI brain a 30% vote


print("Running hyper-fast search with AI Content Matrix...")
hyper_fast_weight_search(
    train_matrix=train_data_matrix_decay,
    item_sim_matrix=item_similarity_sparse,
    content_sim_matrix=content_similarity,
    weight_combos=my_weight_tests,
    k=50,
    batch_size=250
)

In [ ]:
# @title Embeddings model: no cosine sim.
def hyper_fast_weight_search(train_matrix, item_sim_matrix, content_sim_matrix, weight_combos, k=50, batch_size=250):
    n_users = train_matrix.shape[0]

    # Dictionary to hold the lists of recommendations for EACH weight combo
    all_recs = {weights: [] for weights in weight_combos}

    print("Calculating global item popularity for cold-start users...")
    item_popularity = np.array(train_matrix.sum(axis=0)).flatten()
    top_global_items = np.argsort(item_popularity)[-10:][::-1]
    top_global_string = " ".join(map(str, top_global_items))

    # Pre-calculate denominators safely for sparse matrices
    if hasattr(item_sim_matrix, 'toarray'):
        item_sim_sum = np.array(np.abs(item_sim_matrix).sum(axis=1)).flatten() + 1e-9
    else:
        item_sim_sum = np.array(np.abs(item_sim_matrix).sum(axis=1)).flatten() + 1e-9

    content_sim_sum = np.array(np.abs(content_sim_matrix).sum(axis=1)).flatten() + 1e-9

    print(f"Starting hyper-fast grid search for {len(weight_combos)} combinations...")

    for start_idx in range(0, n_users, batch_size):
        end_idx = min(start_idx + batch_size, n_users)
        train_batch = train_matrix[start_idx:end_idx]

        # ==========================================
        # 1. THE HEAVY MATH (Calculated ONLY ONCE per batch)
        # ==========================================

        # U2U: Replaced cosine_similarity with pure sparse Dot Product
        # .T transposes the matrix to align user vectors.
        # .toarray() ensures your k-NN sorting loop below still works.
        u2u_sim_batch = train_batch.dot(train_matrix.T)
        if hasattr(u2u_sim_batch, 'toarray'):
            u2u_sim_batch = u2u_sim_batch.toarray()

        for i in range(u2u_sim_batch.shape[0]):
            row = u2u_sim_batch[i]
            if len(row) > k + 1:
                threshold_idx = np.argpartition(row, -(k+1))[-(k+1)]
                threshold_val = row[threshold_idx]
                row[row < threshold_val] = 0

        u2u_sim_sparse = sparse.csr_matrix(u2u_sim_batch)
        u2u_weighted_sum = u2u_sim_sparse.dot(train_matrix)
        u2u_sum_sim = np.array(u2u_sim_sparse.sum(axis=1)).flatten() + 1e-9

        if hasattr(u2u_weighted_sum, 'toarray'):
            u2u_pred = u2u_weighted_sum.toarray() / u2u_sum_sim[:, None]
        else:
            u2u_pred = u2u_weighted_sum / u2u_sum_sim[:, None]

        # I2I
        i2i_weighted_sum = train_batch.dot(item_sim_matrix)
        if hasattr(i2i_weighted_sum, 'toarray'):
            i2i_pred = i2i_weighted_sum.toarray() / item_sim_sum
        else:
            i2i_pred = i2i_weighted_sum / item_sim_sum

        # CONTENT (AI Embeddings)
        content_weighted_sum = train_batch.dot(content_sim_matrix)
        if hasattr(content_weighted_sum, 'toarray'):
            content_pred = content_weighted_sum.toarray() / content_sim_sum
        else:
            content_pred = content_weighted_sum / content_sim_sum

        # ==========================================
        # 2. THE FAST WEIGHT LOOP
        # ==========================================
        for (w_u, w_i, w_c) in weight_combos:

            ensemble_pred = (u2u_pred * w_u) + (i2i_pred * w_i) + (content_pred * w_c)
            top_10_indices = np.argsort(ensemble_pred, axis=1)[:, -10:][:, ::-1]

            for i in range(top_10_indices.shape[0]):
                if np.max(ensemble_pred[i]) == 0:
                    all_recs[(w_u, w_i, w_c)].append(top_global_string)
                else:
                    recs = " ".join(map(str, top_10_indices[i]))
                    all_recs[(w_u, w_i, w_c)].append(recs)

        # ==========================================
        # 3. RAM CLEANUP
        # ==========================================
        del u2u_sim_batch, u2u_sim_sparse, u2u_weighted_sum, u2u_pred
        del i2i_weighted_sum, i2i_pred, content_weighted_sum, content_pred
        try: del ensemble_pred, top_10_indices
        except: pass
        gc.collect()

    print("\nBuilding submission files...")
    for weights, recs in all_recs.items():
        filename = f"ai_ensemble_{weights[0]}_{weights[1]}_{weights[2]}.csv"
        df_submission = pd.DataFrame({
            'user_id': range(len(recs)),
            'recommendation': recs
        })
        df_submission.to_csv(filename, index=False)
        print(f"File saved successfully: {filename}")

        # Try to trigger the download!
        try:
            from google.colab import files
            files.download(filename)
        except:
            pass

# We assume train_data_matrix_decay, item_sim_sparse_decay, and content_similarity (the embeddings) exist!

my_weight_tests = [
    (0.4, 0.4, 0.2),  # The old baseline (just to see how much the AI improves it alone)
    (0.35, 0.35, 0.3) ] # Giving the AI brain a 30% vote


print("Running hyper-fast search with AI Content Matrix...")
hyper_fast_weight_search(
    train_matrix=train_data_matrix_decay,        # Your time-weighted matrix
    item_sim_matrix=item_similarity_sparse,
    content_sim_matrix=content_similarity,       # Your super-smart SentenceTransformer matrix
    weight_combos=my_weight_tests,
    k=50,
    batch_size=250
)